# Maneuver and Driving-Style Clustering Workbench

Use this notebook to refit **maneuver clusters** and **TDBM-style clusters** on the same analysis table, compare feature sets and methods, visualize PCA / t-SNE, and inspect representative samples.

In [1]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'src').exists():
    REPO_ROOT = Path('/fs/nexus-projects/pc_driving/yaghoubi/tail-risk-motion-prediction')
SRC_ROOT = REPO_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from tailrisk_mp.clustering_workbench import (
    DRIVING_STYLE_FEATURE_SETS,
    MANEUVER_FEATURE_SETS,
    TRAJECTORY_LABELS,
    TDBM_CLASSES,
    compute_projection,
    fit_cluster_result,
    representative_examples,
    run_feature_sweep,
)
from tailrisk_mp.scenario_viz import load_scenario_by_id, plot_scenario

sns.set_theme(style='whitegrid')
ARTIFACT_ROOT = REPO_ROOT / 'artifacts' / 'day2'
TABLE_ROOT = ARTIFACT_ROOT / 'tables'
METADATA_ROOT = ARTIFACT_ROOT / 'metadata'


In [ ]:
CONFIG = {
    'dataset': 'av2',
    'split': 'train',
    'mode': 'driving_style_mode',  # or maneuver_mode
    'feature_set_name': 'style_extended' # or maneuver_extended,
    'method': 'kmeans',  # kmeans, gmm, agglomerative
    'transform': 'mixed_robust',  # raw, mixed_robust, mixed_clip8, quantile_normal
    'clip_mode': 'none',  # none, clip8, winsor
    'n_clusters': 3,
    'error_column': 'proxy_fde',
    'tsne_sample_limit': 3000,
    'feature_sweep_methods': ['kmeans', 'gmm', 'agglomerative'],
    'feature_sweep_transforms': ['mixed_robust', 'mixed_clip8', 'quantile_normal'],
    'feature_sweep_clip_modes': ['none', 'clip8'],
    'feature_sweep_cluster_counts': [3, 4, 5],
    'cluster_id': 0,
    'example_mode': 'nearest',  # nearest, hardest, easiest
    'example_rank': 0,
    'scenario_id': None,
    'view_radius': 90.0,
    'show_track_ids': False,
}

FEATURE_SETS = {
    'maneuver_mode': MANEUVER_FEATURE_SETS,
    'driving_style_mode': DRIVING_STYLE_FEATURE_SETS,
}


In [ ]:
table_path = TABLE_ROOT / f"{CONFIG['dataset']}_{CONFIG['split']}_analysis.csv"
metadata_path = METADATA_ROOT / f"{CONFIG['dataset']}_style_metadata.json"
df = pd.read_csv(table_path)
metadata = json.loads(metadata_path.read_text())

if 'maneuver_cluster_id' not in df.columns and 'style_cluster_id' in df.columns:
    df['maneuver_cluster_id'] = df['style_cluster_id']
if 'maneuver_cluster_distance' not in df.columns and 'cluster_distance' in df.columns:
    df['maneuver_cluster_distance'] = df['cluster_distance']
if 'tdbm_style_label' not in df.columns and 'tdbm_style_id' in df.columns:
    df['tdbm_style_label'] = df['tdbm_style_id'].map(lambda x: TDBM_CLASSES.get(int(x), str(x)))
df['trajectory_label'] = df['trajectory_type'].map(lambda x: TRAJECTORY_LABELS.get(int(x), str(x)))

feature_sets = FEATURE_SETS[CONFIG['mode']]
feature_names = [name for name in feature_sets[CONFIG['feature_set_name']] if name in df.columns]
display(Markdown(f"**Rows:** {len(df):,}  \\n**Feature set:** `{CONFIG['feature_set_name']}`  \\n**Features:** {feature_names}"))
display(df[['scenario_id', 'center_objects_id', 'trajectory_label', 'tdbm_style_label', CONFIG['error_column']]].head())


In [ ]:
result = fit_cluster_result(
    df,
    feature_names,
    method=CONFIG['method'],
    n_clusters=CONFIG['n_clusters'],
    transform_mode=CONFIG['transform'],
    clip_mode=CONFIG['clip_mode'],
    error_column=CONFIG['error_column'],
    reference_trajectory_col='trajectory_type',
    reference_style_col='tdbm_style_id',
)
work_df = df.copy()
work_df['cluster_id'] = result.labels
work_df['cluster_distance'] = result.distances

display(pd.DataFrame([result.diagnostics]).T.rename(columns={0: 'value'}))
display(result.cluster_summary)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
pca_embed, pca_idx = compute_projection(result.transformed[result.feature_names], method='pca')
axes[0].scatter(pca_embed[:, 0], pca_embed[:, 1], c=work_df.iloc[pca_idx]['cluster_id'], cmap='tab10', s=10, alpha=0.6)
axes[0].set_title('PCA')

tsne_embed, tsne_idx = compute_projection(result.transformed[result.feature_names], method='tsne', sample_limit=CONFIG['tsne_sample_limit'])
axes[1].scatter(tsne_embed[:, 0], tsne_embed[:, 1], c=work_df.iloc[tsne_idx]['cluster_id'], cmap='tab10', s=10, alpha=0.6)
axes[1].set_title('t-SNE')
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
traj_support = work_df.pivot_table(index='trajectory_label', columns='cluster_id', values='scenario_id', aggfunc='count', fill_value=0)
traj_error = work_df.pivot_table(index='trajectory_label', columns='cluster_id', values=CONFIG['error_column'], aggfunc='mean')
style_support = work_df.pivot_table(index='tdbm_style_label', columns='cluster_id', values='scenario_id', aggfunc='count', fill_value=0)

sns.heatmap(traj_support, annot=True, fmt='g', cmap='Blues', ax=axes[0])
axes[0].set_title('Support: Trajectory x Cluster')
sns.heatmap(traj_error, annot=True, fmt='.2f', cmap='magma', ax=axes[1])
axes[1].set_title(f"Mean {CONFIG['error_column']}: Trajectory x Cluster")
sns.heatmap(style_support, annot=True, fmt='g', cmap='Greens', ax=axes[2])
axes[2].set_title('Support: TDBM Style x Cluster')
plt.tight_layout()
plt.show()


In [ ]:
sweep = run_feature_sweep(
    df,
    feature_sets,
    methods=CONFIG['feature_sweep_methods'],
    transforms=CONFIG['feature_sweep_transforms'],
    clip_modes=CONFIG['feature_sweep_clip_modes'],
    n_clusters_list=CONFIG['feature_sweep_cluster_counts'],
    error_column=CONFIG['error_column'],
    reference_trajectory_col='trajectory_type',
    reference_style_col='tdbm_style_id',
)
display(sweep.head(20))


In [ ]:
representatives = representative_examples(work_df, result.labels, result.distances, error_column=CONFIG['error_column'])
display(representatives[['cluster_id', 'mode', 'scenario_id', 'center_objects_id', 'trajectory_label', 'tdbm_style_label', CONFIG['error_column'], 'cluster_distance']].head(30))

cluster_df = representatives[representatives['cluster_id'] == CONFIG['cluster_id']]
mode_df = cluster_df[cluster_df['mode'] == CONFIG['example_mode']]
if CONFIG['scenario_id'] is not None:
    candidate = representatives[representatives['scenario_id'] == CONFIG['scenario_id']].iloc[0]
else:
    candidate = mode_df.iloc[min(CONFIG['example_rank'], max(len(mode_df) - 1, 0))] if len(mode_df) else representatives.iloc[0]

split_name = 'train' if CONFIG['split'] == 'train' else ('validation' if CONFIG['dataset'] == 'waymo' else 'val')
bundle = load_scenario_by_id(CONFIG['dataset'], candidate['scenario_id'], split=split_name)
fig, ax = plot_scenario(
    bundle['scenario'],
    dataset=CONFIG['dataset'],
    view_radius=CONFIG['view_radius'],
    show_track_ids=CONFIG['show_track_ids'],
)
plt.show()
display(candidate.to_frame().T)
